# Iteration 3 — per-class threshold tuning

No retraining. Reuses the two checkpoints from before:
`models/baseline_best.pt` (resize pipeline) and `models/crop_best.pt` (crop pipeline).

For each model: get the validation probabilities, then for **each class** sweep the
threshold 0.05 -> 0.95 and keep whichever value maximises that class's F1 on the
validation set. Compare `threshold 0.5` vs `tuned` for both models, on all val
images and on the combo subset.

Purpose: decide whether the crop model's higher noise **PR-AUC** actually converts
into a higher **F1** once the threshold is in the right place — i.e. whether to
keep or revert the crop.

Test split is not touched.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("working dir:", Path.cwd())

import sys
sys.path.append("src")

import numpy as np
import pandas as pd
import torch

from dataset import build_dataloaders
from model import BaselineCNN
from metrics import (evaluate_model, compute_metrics, print_metrics,
                     tune_thresholds, combo_mask, DEFECT_COLUMNS)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

working dir: c:\Users\Shamik\Desktop\ImageQuality_Classifier\ImageQuality_Classifier
device: cuda


## 1. Helper — get a checkpoint's validation predictions

Each model must see val data the same way it was trained: the baseline with
`augment=False` (resize), the crop model with `augment=True` (centre crop).

In [2]:
def get_val_probs(ckpt_path, augment):
    loaders = build_dataloaders(batch_size=32, augment=augment)
    m = BaselineCNN().to(device)
    m.load_state_dict(torch.load(ckpt_path))
    y_true, y_prob, _ = evaluate_model(m, loaders["val"], device)
    files = loaders["val"].dataset.df["filename"].values
    return y_true, y_prob, combo_mask(files)

## 2. Run the comparison for both models

In [3]:
results = {}

for label, ckpt, aug in [("baseline", "models/baseline_best.pt", False),
                         ("crop",     "models/crop_best.pt",     True)]:
    y_true, y_prob, cmask = get_val_probs(ckpt, aug)

    # threshold 0.5
    d_all   = compute_metrics(y_true, y_prob)
    d_combo = compute_metrics(y_true[cmask], y_prob[cmask])

    # tuned per-class thresholds (chosen on the FULL val set)
    thr     = tune_thresholds(y_true, y_prob)
    t_all   = compute_metrics(y_true, y_prob, threshold=thr)
    t_combo = compute_metrics(y_true[cmask], y_prob[cmask], threshold=thr)

    results[label] = dict(d_all=d_all, d_combo=d_combo, t_all=t_all, t_combo=t_combo, thr=thr)

    print(f"\n################  {label.upper()}  ################")
    print("tuned thresholds:", dict(zip(DEFECT_COLUMNS, thr.round(2))))
    print_metrics(d_all,   f"{label} VAL all   | threshold 0.5")
    print_metrics(t_all,   f"{label} VAL all   | tuned")
    print_metrics(d_combo, f"{label} VAL combo | threshold 0.5")
    print_metrics(t_combo, f"{label} VAL combo | tuned")

C:\Users\Shamik\AppData\Local\Temp\ipykernel_25516\1622639327.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(ckpt_path))



################  BASELINE  ################
tuned thresholds: {'blur': np.float64(0.15), 'underexposed': np.float64(0.3), 'overexposed': np.float64(0.4), 'noise': np.float64(0.5), 'contrast': np.float64(0.55)}

=== baseline VAL all   | threshold 0.5 ===
macro-F1 0.853 | macro-P 0.888 | macro-R 0.822 | subset-acc 0.788
class            prec  recall     f1  pr_auc    tp    fp    fn    tn
blur            0.906   0.763  0.828   0.911   106    11    33   572
underexposed    0.975   0.902  0.937   0.993   119     3    13   587
overexposed     0.927   0.885  0.906   0.956   115     9    15   583
noise           0.742   0.662  0.700   0.787    92    32    47   551
contrast        0.887   0.900  0.894   0.905   126    16    14   566

=== baseline VAL all   | tuned ===
macro-F1 0.863 | macro-P 0.876 | macro-R 0.852 | subset-acc 0.791
class            prec  recall     f1  pr_auc    tp    fp    fn    tn
blur            0.861   0.849  0.855   0.911   118    19    21   564
underexposed    0.954   

C:\Users\Shamik\AppData\Local\Temp\ipykernel_25516\1622639327.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(ckpt_path))



################  CROP  ################
tuned thresholds: {'blur': np.float64(0.15), 'underexposed': np.float64(0.1), 'overexposed': np.float64(0.85), 'noise': np.float64(0.2), 'contrast': np.float64(0.4)}

=== crop VAL all   | threshold 0.5 ===
macro-F1 0.845 | macro-P 0.919 | macro-R 0.796 | subset-acc 0.799
class            prec  recall     f1  pr_auc    tp    fp    fn    tn
blur            0.990   0.691  0.814   0.931    96     1    43   582
underexposed    0.976   0.932  0.953   0.993   123     3     9   587
overexposed     0.851   0.923  0.886   0.951   120    21    10   571
noise           0.865   0.554  0.675   0.827    77    12    62   571
contrast        0.911   0.879  0.895   0.915   123    12    17   570

=== crop VAL all   | tuned ===
macro-F1 0.876 | macro-P 0.888 | macro-R 0.866 | subset-acc 0.809
class            prec  recall     f1  pr_auc    tp    fp    fn    tn
blur            0.934   0.813  0.869   0.931   113     8    26   575
underexposed    0.930   1.000  0.964

## 3. Side-by-side summary

In [4]:
rows = []
for label in ["baseline", "crop"]:
    r = results[label]
    rows.append({
        "model": label,
        "macroF1_all @0.5":   round(r["d_all"]["macro_f1"], 3),
        "macroF1_all tuned":  round(r["t_all"]["macro_f1"], 3),
        "macroF1_combo @0.5": round(r["d_combo"]["macro_f1"], 3),
        "macroF1_combo tuned":round(r["t_combo"]["macro_f1"], 3),
        "noiseF1_all @0.5":   round(r["d_all"]["per_class"]["noise"]["f1"], 3),
        "noiseF1_all tuned":  round(r["t_all"]["per_class"]["noise"]["f1"], 3),
        "noiseF1_combo tuned":round(r["t_combo"]["per_class"]["noise"]["f1"], 3),
    })
pd.DataFrame(rows).set_index("model")

,macroF1_all @0.5,macroF1_all tuned,macroF1_combo @0.5,macroF1_combo tuned,noiseF1_all @0.5,noiseF1_all tuned,noiseF1_combo tuned
model,,,,,,,
baseline,0.853,0.863,0.53,0.605,0.700,0.700,0.071
crop,0.845,0.876,0.52,0.684,0.675,0.746,0.364


## 4. Read it

- **`crop tuned` macroF1_all clearly beats `baseline tuned`** -> the crop's PR-AUC
  gain was real; keep the crop, adopt the tuned thresholds.
- **`crop tuned` roughly equals `baseline tuned`** -> the crop bought nothing;
  revert to the resize pipeline, and if noise still lags, regenerate noise images
  with a higher sigma floor.
- Either way, note whether tuned thresholds rescued **combo** noise recall (it was
  0 TP at 0.5). If PR-AUC was high but tuning still can't fix combos, that points
  to needing combos in the training set.

Caveat: thresholds picked to maximise val F1 make the tuned val numbers slightly
optimistic. The honest figure is the eventual test-set run with these thresholds
frozen.